
### Cell 1: Setup & Dependencies

In [1]:
# Install required packages for NIfTI processing and deep learning
!pip install -q nibabel scipy torch tqdm --no-deps

import glob
import os
import shutil
import tarfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
import scipy.ndimage as ndimage
import torch
import torch.nn as nn
from tqdm import tqdm

print("Environment setup complete. nibabel version:", nib.__version__)

Environment setup complete. nibabel version: 5.4.2


---

### Cell 2: Dataset Path & Archive Extraction

In [2]:
# Setup paths based on Kaggle input directory
brats_root = "/kaggle/input/datasets/dschettler8845/brats-2021-task1"
brats_extract_dir = "/kaggle/working/brats2021"

os.makedirs(brats_extract_dir, exist_ok=True)

# Unpack .tar archives
tar_files = glob.glob(f"{brats_root}/*.tar")
print(f"Found {len(tar_files)} tar files:", [os.path.basename(t) for t in tar_files])

for tar_path in tar_files:
    print(f"Extracting {os.path.basename(tar_path)} ...")
    with tarfile.open(tar_path, "r") as tf:
        for member in tqdm(tf.getmembers(), desc=os.path.basename(tar_path)):
            tf.extract(member, brats_extract_dir)

n_extracted = sum(len(files) for _, _, files in os.walk(brats_extract_dir))
print("Done. Total files extracted:", n_extracted)

total_gb = sum(os.path.getsize(t) for t in tar_files) / 1e9
print(f"{len(tar_files)} tar files, {total_gb:.1f} GB total (compressed/on-disk size)")

Found 3 tar files: ['BraTS2021_00495.tar', 'BraTS2021_Training_Data.tar', 'BraTS2021_00621.tar']
Extracting BraTS2021_00495.tar ...


BraTS2021_00495.tar:   0%|          | 0/6 [00:00<?, ?it/s]/tmp/ipykernel_58/1574269953.py:15: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extract(member, brats_extract_dir)
BraTS2021_00495.tar: 100%|██████████| 6/6 [00:00<00:00, 44.30it/s]


Extracting BraTS2021_Training_Data.tar ...


BraTS2021_Training_Data.tar: 100%|██████████| 7508/7508 [01:13<00:00, 102.12it/s]


Extracting BraTS2021_00621.tar ...


BraTS2021_00621.tar: 100%|██████████| 6/6 [00:00<00:00, 115.77it/s]


Done. Total files extracted: 6266
3 tar files, 13.4 GB total (compressed/on-disk size)


---

### Cell 3: Data Filtering & Outlier Removal from Disk

In [3]:
# Scan all extracted paths, keeping only actual directories (filters out loose files)
all_patient_folders = [f for f in glob.glob(os.path.join(brats_extract_dir, "BraTS2021_*")) if os.path.isdir(f)]

# Flag known duplicate and extreme outlier IDs identified during EDA
invalid_cases = {
    "BraTS2021_00495",  # Duplicate scan
    "BraTS2021_00621",  # Duplicate scan
    "BraTS2021_01163"   # Outlier: 1000x brightness anomaly
}

clean_dataset_paths = []
for folder in all_patient_folders:
    patient_id = os.path.basename(folder)
    if patient_id in invalid_cases:
        # Delete dropped outlier directories directly from disk
        shutil.rmtree(folder)
        print(f"Removed outlier directory: {patient_id}")
    else:
        clean_dataset_paths.append(folder)

print(f"Total valid patient directories: {len(all_patient_folders)}")
print(f"Clean, usable cases remaining: {len(clean_dataset_paths)}")

Removed outlier directory: BraTS2021_01163
Removed outlier directory: BraTS2021_00621
Removed outlier directory: BraTS2021_00495
Total valid patient directories: 1251
Clean, usable cases remaining: 1248


---

### Cell 4: Intensity Normalization Logic

In [4]:
def preprocess_mri_volume(volume_data, lower_percentile=1.0, upper_percentile=99.0):
    """
    1. Winsorizes (clips) extreme brightness values to clear scanning artifacts.
    2. Performs Z-score normalization restricted strictly to non-zero brain tissue.
    """
    brain_mask = volume_data > 0
    if not brain_mask.any():
        return volume_data.astype(np.float32)

    # 1. Clip extreme percentiles inside the brain tissue
    brain_voxels = volume_data[brain_mask]
    low_val, high_val = np.percentile(brain_voxels, [lower_percentile, upper_percentile])
    clipped_data = np.clip(volume_data, low_val, high_val)

    # 2. Z-Score normalization on brain tissue only (mean=0, std=1)
    mean_val = np.mean(clipped_data[brain_mask])
    std_val = np.std(clipped_data[brain_mask])

    if std_val < 1e-8:
        std_val = 1.0

    normalized_data = np.zeros_like(volume_data, dtype=np.float32)
    normalized_data[brain_mask] = (clipped_data[brain_mask] - mean_val) / std_val

    return normalized_data

print("Intensity normalization function defined.")

Intensity normalization function defined.


---

### Cell 5: Clinical Metrics & Multifocality Correction

In [5]:
def extract_clinical_metrics(segmentation_mask, voxel_volume_mm3=1.0, min_foci_voxels=500):
    """
    1. Removes tiny noise fragments using 3D connected component labeling
       to rectify inflated multifocality counts.
    2. Calculates true solid tumor volume by excluding Edema (Label 2).
    """
    # --- 1. Correct Multifocality Count ---
    binary_tumor_mask = (segmentation_mask > 0).astype(np.uint8)
    labeled_array, num_features = ndimage.label(binary_tumor_mask)

    true_foci_count = 0
    cleaned_mask = np.zeros_like(segmentation_mask)

    for cluster_id in range(1, num_features + 1):
        cluster_voxels = (labeled_array == cluster_id)
        if np.sum(cluster_voxels) >= min_foci_voxels:
            true_foci_count += 1
            cleaned_mask[cluster_voxels] = segmentation_mask[cluster_voxels]

    # --- 2. Calculate Solid Tumor Size ---
    # BraTS Labels: 1 = Necrotic Core, 2 = Edema (Swelling), 4 = Enhancing Tumor
    # Exclude Label 2 to track solid tumor instead of swelling
    solid_mask = np.logical_or(cleaned_mask == 1, cleaned_mask == 4)
    solid_volume_mm3 = np.sum(solid_mask) * voxel_volume_mm3

    return true_foci_count, solid_volume_mm3, cleaned_mask

print("Clinical metric extraction functions defined.")

Clinical metric extraction functions defined.


---

### Cell 6: Generalized Dice Loss (Class Imbalance)

In [6]:
class GeneralizedDiceLoss(nn.Module):
    """
    Solves extreme class imbalance by weighting the spatial overlap
    inversely to class volume.
    """
    def __init__(self, epsilon=1e-6):
        super(GeneralizedDiceLoss, self).__init__()
        self.epsilon = epsilon

    def forward(self, predictions, targets):
        # Expected shape: (Batch, Classes, Depth, Height, Width)
        class_volumes = torch.sum(targets, dim=(0, 2, 3, 4))
        weights = 1.0 / ((class_volumes ** 2) + self.epsilon)

        intersection = torch.sum(predictions * targets, dim=(0, 2, 3, 4))
        union = torch.sum(predictions + targets, dim=(0, 2, 3, 4))

        dice_score = 2.0 * torch.sum(weights * intersection) / torch.sum(weights * union + self.epsilon)
        return 1.0 - dice_score

print("Generalized Dice Loss function ready.")

Generalized Dice Loss function ready.


---

### Cell 7: Process NIfTI Scans (Apply Normalization)

In [ ]:
# Iterate through clean patient folders and overwrite original files with processed scans
print("Applying preprocessing (clipping + Z-score) to all NIfTI MRI volumes...")

for folder in tqdm(clean_dataset_paths, desc="Preprocessing Scans"):
    for file_name in os.listdir(folder):
        # Process MRI modality scans (flair, t1, t1ce, t2); exclude segmentation masks from intensity scaling
        if file_name.endswith(".nii.gz") and not file_name.endswith("_seg.nii.gz"):
            file_path = os.path.join(folder, file_name)

            # Load NIfTI volume
            nii_img = nib.load(file_path)
            volume_data = nii_img.get_fdata()

            # Apply intensity clipping and Z-score normalization
            processed_data = preprocess_mri_volume(volume_data)

            # Overwrite original file with preprocessed array
            processed_img = nib.Nifti1Image(processed_data, nii_img.affine, nii_img.header)
            nib.save(processed_img, file_path)

print("Preprocessing complete! All MRI volumes have been normalized on disk.")

Applying preprocessing (clipping + Z-score) to all NIfTI MRI volumes...


Preprocessing Scans:  74%|███████▍  | 924/1248 [42:08<13:41,  2.54s/it]  

---

### Cell 8: Archive & Disk Cleanup

In [ ]:
# 1. Define source and destination paths clearly
source_dir = "/kaggle/working/brats2021"
archive_basename = "/kaggle/working/brats2021_processed"

print(f"Archiving preprocessed dataset from {source_dir}...")

# 2. Archive only the contents of source_dir
shutil.make_archive(
    base_name=archive_basename,
    format='tar',
    root_dir=source_dir
)

archive_file = archive_basename + ".tar"
print(f"Archive successfully created: {archive_file}")
print(f"Size: {os.path.getsize(archive_file) / 1e9:.2f} GB")

# 3. Delete uncompressed raw directory to stay under Kaggle disk limits
print("Cleaning up uncompressed working directory...")
shutil.rmtree(source_dir)
print("Cleanup complete! 'brats2021_processed.tar' is ready to mount in the next notebook.")